# RAG: 검색증강생성 완전 가이드 - 실습 코드 1: RAG 시스템 구현 (LangChain)

- Tutorial ID: `expand-rag-fundamentals`
- Tutorial: RAG: 검색증강생성 완전 가이드
- Section ID: `expand-rag-fundamentals-code-1`
- Section: 실습 코드 1: RAG 시스템 구현 (LangChain)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: RAG 시스템 구현 (LangChain)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 질문 → 검색 → 근거 선택 → 답변/검증 파이프라인을 단계별로 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI

# 1. 문서 로딩
loader = TextLoader("knowledge_base.txt")
docs = loader.load()

# 2. 청킹
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(docs)

# 3. 임베딩 & 인덱싱
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma.from_documents(
    chunks, embeddings, persist_directory="./db"
)

# 4. RAG 체인 구성
qa_chain = RetrievalQA.from_chain_type(
    llm=OpenAI(temperature=0),
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
)

# 5. 질의
result = qa_chain({"query": "What are the key benefits of RAG?"})
print(f"Answer: {result['result']}")
print(f"Sources: {[d.metadata for d in result['source_documents']]}")